# TornadoWatch — Training Notebook

**Goal:** Train a Transformer model to predict tornado probability at a coordinate 1–2 hours ahead.  
**Data:** SPC storm reports (labels) + ERA5 reanalysis via Open-Meteo (features)  
**Pipeline:**
1. Data download & EDA
2. Feature engineering & dataset construction
3. Phase 1 — Backbone pretraining (atmospheric forecasting)
4. Phase 2 — Fine-tuning on tornado labels
5. Evaluation (POD, FAR, CSI, AUC-ROC, calibration)
6. Export to serving checkpoint

In [ ]:
import sys
import logging
sys.path.insert(0, '../backend')

from dotenv import load_dotenv
load_dotenv('../backend/.env')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(name)s %(levelname)s: %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)

import asyncio
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
from datetime import datetime, timezone

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore', category=UserWarning)
plt.style.use('dark_background')
sns.set_palette('husl')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

DATA_DIR = Path('../backend/data')
CKPT_DIR = Path('../backend/checkpoints')
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

---
## 1. Download SPC Tornado Reports

SPC publishes annual CSV files of all tornado reports including lat/lon, EF scale, time, path length.  
We download multiple years to build a statistically meaningful dataset.

In [ ]:
# Sanity check — verify Open-Meteo customer API and SPC are reachable
import httpx, os
from datetime import timedelta
from dotenv import load_dotenv
load_dotenv("../backend/.env")

async def _era5_test():
    from ml.data import fetch_era5_features
    end = datetime(2023, 5, 6, 20, 0, tzinfo=timezone.utc)
    start = end - timedelta(hours=24)
    async with httpx.AsyncClient() as client:
        feats = await fetch_era5_features(35.5, -97.5, start, end, client)
    if feats is not None and len(feats) >= 6:
        nan_pct = np.isnan(feats).mean() * 100
        print(f"Open-Meteo OK — {len(feats)} hourly rows, {feats.shape[1]} features, {nan_pct:.1f}% NaN")
        print(f"  CAPE range:  {feats[:,0].min():.1f} - {feats[:,0].max():.1f} J/kg")
        print(f"  Shear range: {feats[:,4].min():.1f} - {feats[:,4].max():.1f} m/s")
    else:
        print("Open-Meteo FAILED — check API key in .env")
    return feats

async def _spc_test():
    from ml.data import fetch_spc_tornado_reports
    async with httpx.AsyncClient() as client:
        try:
            df = await fetch_spc_tornado_reports(2023, client)
            print(f"SPC API OK — {len(df)} reports for 2023")
        except Exception as e:
            print(f"SPC API FAILED: {e}")

await _spc_test()
test_feats = await _era5_test()

In [ ]:
import httpx
from io import StringIO

SPC_URL = 'https://www.spc.noaa.gov/wcm/data/{year}_torn.csv'
TRAINING_YEARS = list(range(2010, 2024))  # 14 years of data

async def download_spc_reports(years):
    all_dfs = []
    cache = DATA_DIR / 'spc_reports.parquet'
    if cache.exists():
        print(f'Loading cached SPC reports from {cache}')
        return pd.read_parquet(cache)

    async with httpx.AsyncClient(timeout=30) as client:
        for year in years:
            try:
                r = await client.get(SPC_URL.format(year=year))
                r.raise_for_status()
                df = pd.read_csv(StringIO(r.text), encoding='latin-1')
                df['source_year'] = year
                all_dfs.append(df)
                print(f'  {year}: {len(df)} reports')
            except Exception as e:
                print(f'  {year}: FAILED — {e}')

    combined = pd.concat(all_dfs, ignore_index=True)

    # Standardize columns
    combined = combined.rename(columns={'slat': 'lat', 'slon': 'lon'})
    ef_col = 'mag' if 'mag' in combined.columns else 'f'
    combined['ef_scale'] = pd.to_numeric(combined[ef_col], errors='coerce').fillna(0).astype(int)
    combined['lat'] = pd.to_numeric(combined['lat'], errors='coerce')
    combined['lon'] = pd.to_numeric(combined['lon'], errors='coerce')
    combined = combined.dropna(subset=['lat', 'lon'])
    combined = combined[(combined['lat'] != 0) & (combined['lon'] != 0)]

    # SPC time column is HH:MM:SS — use the date column directly
    combined['datetime'] = pd.to_datetime(
        combined['date'].astype(str) + ' ' + combined['time'].astype(str),
        errors='coerce', utc=True
    )

    combined = combined[['datetime', 'lat', 'lon', 'ef_scale', 'source_year']].dropna()
    combined.to_parquet(cache)
    return combined

print('Downloading SPC tornado reports...')
reports = await download_spc_reports(TRAINING_YEARS)
print(f'\nTotal tornado reports: {len(reports):,}')

---
## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('SPC Tornado Reports — EDA', fontsize=14, color='white')

# Annual count
ax = axes[0, 0]
yearly = reports.groupby('source_year').size()
ax.bar(yearly.index, yearly.values, color='#38bdf8', alpha=0.8)
ax.set_title('Annual Tornado Count')
ax.set_xlabel('Year')
ax.set_ylabel('Count')

# EF scale distribution
ax = axes[0, 1]
ef_counts = reports['ef_scale'].value_counts().sort_index()
colors = ['#10b981', '#eab308', '#f97316', '#ef4444', '#7f1d1d', '#450a0a']
ax.bar(ef_counts.index, ef_counts.values, color=colors[:len(ef_counts)], alpha=0.9)
ax.set_title('EF Scale Distribution')
ax.set_xlabel('EF Scale')
ax.set_ylabel('Count')
ax.set_yscale('log')

# Monthly distribution
ax = axes[0, 2]
reports['month'] = pd.to_datetime(reports['datetime']).dt.month
monthly = reports.groupby('month').size()
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
ax.bar(range(1, 13), monthly.reindex(range(1, 13), fill_value=0).values, color='#a78bfa', alpha=0.8)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months, rotation=45, fontsize=8)
ax.set_title('Monthly Distribution')

# Hour of day
ax = axes[1, 0]
reports['hour'] = pd.to_datetime(reports['datetime']).dt.hour
hourly = reports.groupby('hour').size()
ax.plot(hourly.index, hourly.values, color='#f59e0b', linewidth=2)
ax.fill_between(hourly.index, hourly.values, alpha=0.3, color='#f59e0b')
ax.set_title('Hour of Day (UTC)')
ax.set_xlabel('Hour')

# Geographic distribution
ax = axes[1, 1]
conus = reports[(reports['lat'].between(25, 50)) & (reports['lon'].between(-125, -65))]
ax.hexbin(conus['lon'], conus['lat'], gridsize=40, cmap='YlOrRd', mincnt=1)
ax.set_title('Geographic Density (CONUS)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# EF3+ events only (significant tornadoes)
ax = axes[1, 2]
sig = reports[reports['ef_scale'] >= 3]
conus_sig = sig[(sig['lat'].between(25, 50)) & (sig['lon'].between(-125, -65))]
ax.hexbin(conus_sig['lon'], conus_sig['lat'], gridsize=30, cmap='Reds', mincnt=1)
ax.set_title('EF3+ Significant Tornadoes')
ax.set_xlabel('Longitude')

plt.tight_layout()
plt.savefig(DATA_DIR / 'eda.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nKey stats:')
print(f'  Total events:     {len(reports):,}')
print(f'  EF0:              {(reports.ef_scale==0).sum():,}  ({(reports.ef_scale==0).mean()*100:.1f}%)')
print(f'  EF1:              {(reports.ef_scale==1).sum():,}  ({(reports.ef_scale==1).mean()*100:.1f}%)')
print(f'  EF2+:             {(reports.ef_scale>=2).sum():,}  ({(reports.ef_scale>=2).mean()*100:.1f}%)')
print(f'  EF3+ (violent):   {(reports.ef_scale>=3).sum():,}  ({(reports.ef_scale>=3).mean()*100:.1f}%)')
print(f'  Date range:       {reports.datetime.min().date()} → {reports.datetime.max().date()}')

---
## 3. Build Training Dataset

For each tornado event, we fetch 24h of ERA5 atmospheric data leading up to it (positive sample).  
Negative samples are drawn from random CONUS locations on non-event days at a 3:1 ratio.

> **Runtime note:** First run downloads ~10GB of ERA5 data and takes several hours. Results are cached to `data/cache/`.

In [ ]:
from ml.data import build_dataset, AtmosphericScaler, TornadoDataset
from ml.model import FEATURE_NAMES, N_FEATURES
import pickle

SCALER_PATH = CKPT_DIR / 'scaler.pkl'

# Temporal split — always predict forward, never backward.
# Train: 2015-2023 | Val: 2024 | Test: 2025 (held-out, never touched during training)
TRAIN_END_YEAR = 2023
VAL_YEAR       = 2024
TEST_YEAR      = 2025

COMBINED_CACHE = DATA_DIR / 'cache/samples_2015_2025.pkl'

if COMBINED_CACHE.exists():
    print(f'Loading combined cache...')
    with open(COMBINED_CACHE, 'rb') as f:
        all_samples = pickle.load(f)
    print(f'Loaded {len(all_samples):,} samples from {COMBINED_CACHE}')
else:
    raise FileNotFoundError(f'Combined cache not found: {COMBINED_CACHE}. Run build_dataset first.')

# Year-based splits — respect temporal ordering
samples       = [s for s in all_samples if s.valid_time.year <= TRAIN_END_YEAR]
val_samples   = [s for s in all_samples if s.valid_time.year == VAL_YEAR]
test_samples  = [s for s in all_samples if s.valid_time.year == TEST_YEAR]

pos       = sum(s.label for s in samples)
neg       = len(samples) - pos
years_covered = sorted(set(s.valid_time.year for s in samples))

print(f'\nDataset splits:')
print(f'  Train (≤{TRAIN_END_YEAR}):  {len(samples):,} samples  '
      f'pos={pos:,} ({pos/len(samples)*100:.1f}%)  years={years_covered}')
print(f'  Val  ({VAL_YEAR}):   {len(val_samples):,} samples  '
      f'pos={sum(s.label for s in val_samples):,} '
      f'({sum(s.label for s in val_samples)/len(val_samples)*100:.1f}%)')
print(f'  Test ({TEST_YEAR}):   {len(test_samples):,} samples  '
      f'pos={sum(s.label for s in test_samples):,} '
      f'({sum(s.label for s in test_samples)/len(test_samples)*100:.1f}%)')
print(f'  Class imbalance (train): 1:{neg//pos}')


In [ ]:
# ── Patch EF scale into existing sample caches ───────────────────────────────
# Reads SPC CSVs (fast, no ERA5 calls) and patches ef_scale onto cached samples.
# Overwrites cache files in place so downstream cells pick up real EF values.
import pickle, httpx
from io import StringIO
from pathlib import Path
import pandas as pd

DATA_DIR = Path('../backend/data')
CACHE_DIR = DATA_DIR / 'cache'

# Download SPC reports for all years in our caches
SPC_URL = 'https://www.spc.noaa.gov/wcm/data/{year}_torn.csv'
ALL_YEARS = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

print('Fetching SPC reports to build EF lookup...')
dfs = []
async with httpx.AsyncClient(timeout=30) as client:
    for year in ALL_YEARS:
        try:
            r = await client.get(SPC_URL.format(year=year), follow_redirects=True)
            r.raise_for_status()
            df = pd.read_csv(StringIO(r.text), encoding='latin-1')
            df = df.rename(columns={'slat': 'lat', 'slon': 'lon'})
            ef_col = 'mag' if 'mag' in df.columns else 'f'
            df['ef_scale'] = pd.to_numeric(df[ef_col], errors='coerce').fillna(0)
            df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
            df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
            df['datetime'] = pd.to_datetime(
                df['date'].astype(str) + ' ' + df['time'].astype(str),
                errors='coerce', utc=True
            )
            df = df[['datetime', 'lat', 'lon', 'ef_scale']].dropna()
            dfs.append(df)
            print(f'  {year}: {len(df)} reports')
        except Exception as e:
            print(f'  {year}: FAILED — {e}')

spc = pd.concat(dfs, ignore_index=True)

# Build lookup: (lat_3dp, lon_3dp, iso_datetime) → ef_scale
ef_lookup = {}
for _, row in spc.iterrows():
    key = (round(float(row['lat']), 3), round(float(row['lon']), 3), row['datetime'].isoformat())
    ef_lookup[key] = float(row['ef_scale'])
print(f'\nEF lookup built: {len(ef_lookup):,} events')

# Patch all cache files
cache_files = list(CACHE_DIR.glob('samples_*.pkl'))
for cache_path in sorted(cache_files):
    with open(cache_path, 'rb') as f:
        samples = pickle.load(f)
    patched, already_set = 0, 0
    for s in samples:
        if s.label == 1:
            key = (round(s.lat, 3), round(s.lon, 3), s.valid_time.isoformat())
            ef = ef_lookup.get(key, 0.0)
            if hasattr(s, 'ef_scale') and s.ef_scale != 0.0:
                already_set += 1
            else:
                s.ef_scale = ef
                patched += 1
        else:
            s.ef_scale = 0.0
    with open(cache_path, 'wb') as f:
        pickle.dump(samples, f)
    pos = sum(s.label for s in samples)
    ef_nonzero = sum(1 for s in samples if s.label == 1 and getattr(s, 'ef_scale', 0) > 0)
    print(f'  {cache_path.name}: {len(samples):,} samples, {patched} patched, '
          f'{ef_nonzero}/{pos} positives have EF>0')

print('\nEF scale patch complete. Re-run scaler and model config cells before retraining.')


In [ ]:
# Fit and save scaler
all_features = np.stack([s.features for s in samples])
scaler = AtmosphericScaler()
scaled_features = scaler.fit_transform(all_features)
scaler.save(SCALER_PATH)
print(f'Scaler saved to {SCALER_PATH}')

# Feature statistics
feat_means = all_features.mean(axis=(0,1))
feat_stds  = all_features.std(axis=(0,1))

fig, ax = plt.subplots(figsize=(14, 4))
x = np.arange(N_FEATURES)
ax.bar(x - 0.2, feat_means, 0.35, label='Mean', color='#38bdf8', alpha=0.8)
ax.bar(x + 0.2, feat_stds,  0.35, label='Std',  color='#f59e0b', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_title('Feature Statistics (pre-normalization)')
ax.legend()
plt.tight_layout()
plt.show()

# Positive vs negative feature distributions for key variables
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Feature Distributions: Tornado vs. No-Tornado', fontsize=13)
key_features = ['cape', 'srh_03km', 'shear_06km', 'lifted_index', 'cin', 'temp_2m']
pos_feats = all_features[[i for i,s in enumerate(samples) if s.label==1]]
neg_feats = all_features[[i for i,s in enumerate(samples) if s.label==0]]

for ax, feat in zip(axes.flat, key_features):
    idx = FEATURE_NAMES.index(feat)
    p = pos_feats[:, -1, idx]
    n = neg_feats[:, -1, idx]
    p = p[np.isfinite(p)]
    n = n[np.isfinite(n)]
    bins = np.linspace(np.percentile(np.concatenate([p,n]), 2),
                       np.percentile(np.concatenate([p,n]), 98), 40)
    ax.hist(n, bins=bins, alpha=0.6, label='No tornado', color='#38bdf8', density=True)
    ax.hist(p, bins=bins, alpha=0.6, label='Tornado',    color='#ef4444', density=True)
    ax.set_title(feat)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

---
## 4. Phase 1 — Backbone Pretraining

Train the Transformer backbone to predict the next time step's atmospheric state.  
This gives the model a physical "prior" before it ever sees a tornado label.

In [ ]:
from ml.model import TornadoTransformer, ModelConfig, INTENSITY_CLASS_NAMES
from ml.data import TornadoDataset
from torch.utils.data import DataLoader

cfg = ModelConfig(
    n_features=N_FEATURES,
    seq_len=24,
    d_model=256,
    n_heads=8,
    n_encoder_layers=6,
    d_ff=1024,
    dropout=0.1,
    forecast_hours=2,
    n_intensity_classes=3,  # 0=None, 1=Weak (EF0-2), 2=Significant (EF3+)
)

# Build DataLoaders from year-based splits
train_ds = TornadoDataset(samples, scaler)
val_ds   = TornadoDataset(val_samples, scaler)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=0)

model = TornadoTransformer(cfg).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,} total, {trainable:,} trainable')
print(f'Train samples: {len(train_ds):,} | Val samples: {len(val_ds):,}')
print(f'Train covers: {train_ds.samples[0].valid_time.date()} → {train_ds.samples[-1].valid_time.date()}')
print(f'Val covers:   {val_ds.samples[0].valid_time.date()} → {val_ds.samples[-1].valid_time.date()}')
print(f'Intensity classes: {INTENSITY_CLASS_NAMES}')


In [ ]:
PRETRAIN_EPOCHS = 30
pretrain_losses = []

optimizer  = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PRETRAIN_EPOCHS)
criterion  = nn.MSELoss()

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        x = batch['features'].to(DEVICE)
        inp, target = x[:, :-1, :], x[:, 1:, :]
        optimizer.zero_grad()
        pred = model.backbone.pretrain_forward(inp)
        loss = criterion(pred, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    avg = epoch_loss / len(train_loader)
    pretrain_losses.append(avg)
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{PRETRAIN_EPOCHS} — loss: {avg:.5f}  lr: {scheduler.get_last_lr()[0]:.2e}')

model.save(CKPT_DIR / 'pretrained_best.pt', {'epochs': PRETRAIN_EPOCHS, 'pretrain_loss': pretrain_losses[-1]})
print(f'\nPretrained checkpoint saved.')

# Loss curve
plt.figure(figsize=(8, 3))
plt.plot(pretrain_losses, color='#38bdf8', linewidth=2)
plt.title('Pretraining Loss (next-step atmospheric prediction)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.tight_layout()
plt.show()

In [ ]:
# Load pretrained checkpoint — skip retraining Phase 1
model.load_state_dict(torch.load(CKPT_DIR / 'pretrained_best.pt', weights_only=False)['state_dict'])     
print('Pretrained checkpoint loaded.')

---
## 5. Phase 2 — Fine-Tuning on Tornado Labels

Two sub-phases:
- **2a:** Freeze backbone, train tornado head only (fast convergence, prevents catastrophic forgetting)
- **2b:** Progressively unfreeze last N backbone layers + head (full adaptation)

In [ ]:
from torch.utils.data import WeightedRandomSampler

# Weighted sampler to handle class imbalance
labels = [s.label for s in train_ds.samples]
n_pos, n_neg = sum(labels), len(labels) - sum(labels)
w_pos, w_neg = 1.0/n_pos, 1.0/n_neg
weights = [w_pos if l==1 else w_neg for l in labels]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
balanced_loader = DataLoader(train_ds, batch_size=64, sampler=sampler, num_workers=0)

bce = nn.BCELoss()

def run_epoch(loader, train=True, opt=None):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            x = batch['features'].to(DEVICE)
            y = batch['label'].to(DEVICE)
            out = model(x)
            prob = out['tornado_prob'].mean(dim=1)
            loss = bce(prob, y)
            if train and opt:
                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            total_loss += loss.item()
            preds = (prob > 0.5).float()
            correct += (preds == y).sum().item()
            total += len(y)
            all_probs.extend(prob.detach().cpu().tolist())
            all_labels.extend(y.cpu().tolist())
    return {
        'loss': total_loss / len(loader),
        'acc': correct / total,
        'probs': np.array(all_probs),
        'labels': np.array(all_labels),
    }

# Phase 2a — head only
print('Phase 2a: Training tornado head (backbone frozen)...')
model.freeze_backbone()
opt_head = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
)

phase2a_log = []
for epoch in range(1, 11):
    tr = run_epoch(balanced_loader, train=True, opt=opt_head)
    vl = run_epoch(val_loader, train=False)
    phase2a_log.append({'epoch': epoch, 'train_loss': tr['loss'], 'val_loss': vl['loss'], 'val_acc': vl['acc']})
    print(f'  Epoch {epoch:2d}/10 — train_loss:{tr["loss"]:.4f}  val_loss:{vl["loss"]:.4f}  val_acc:{vl["acc"]*100:.1f}%')

In [ ]:
from torch.utils.data import WeightedRandomSampler
from ml.model import INTENSITY_CLASS_NAMES

# Weighted sampler for binary class imbalance
labels = [s.label for s in train_ds.samples]
n_pos, n_neg = sum(labels), len(labels) - sum(labels)
w_pos, w_neg = 1.0/n_pos, 1.0/n_neg
weights = [w_pos if l==1 else w_neg for l in labels]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
balanced_loader = DataLoader(train_ds, batch_size=64, sampler=sampler, num_workers=0)

bce = nn.BCELoss()

# Intensity class weights — inverse frequency to handle EF3+ rarity
# Class 0 (None): ~76%  Class 1 (Weak): ~23%  Class 2 (Significant): ~1%
intensity_class_weights = torch.tensor([0.10, 0.40, 5.0], device=DEVICE)
ce_loss = nn.CrossEntropyLoss(weight=intensity_class_weights)


def ef_to_intensity_class(ef_scale: torch.Tensor, label: torch.Tensor) -> torch.Tensor:
    """Map EF scale + binary label → 3-class intensity label."""
    cls = torch.zeros_like(ef_scale, dtype=torch.long)
    cls = torch.where((label > 0.5) & (ef_scale < 3),  torch.ones_like(cls),     cls)
    cls = torch.where((label > 0.5) & (ef_scale >= 3), torch.full_like(cls, 2),  cls)
    return cls


FINETUNE_EPOCHS     = 40
UNFREEZE_LAYERS     = 2
INTENSITY_LOSS_WEIGHT = 0.30   # higher weight now that it's a real classification task

print(f'Phase 2b: Unfreezing last {UNFREEZE_LAYERS} backbone layers + head...')
model.unfreeze_backbone(UNFREEZE_LAYERS)
trainable_now = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable_now:,}')

opt_full = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=3e-5, weight_decay=1e-4
)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt_full, T_max=FINETUNE_EPOCHS)

best_f1, best_f2 = 0.0, 0.0
finetune_log = []


def run_epoch_intensity(loader, train=True, opt=None):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    intensity_correct, intensity_total = 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            x      = batch['features'].to(DEVICE)
            y      = batch['label'].to(DEVICE)
            ef_tgt = batch['ef_scale'].to(DEVICE)
            out    = model(x)
            prob   = out['tornado_prob'].mean(dim=1)
            # intensity_logits: (batch, forecast_hours, n_classes) → avg hours → (batch, n_classes)
            logits = out['intensity_logits'].mean(dim=1)
            i_cls  = ef_to_intensity_class(ef_tgt, y)

            loss_cls       = bce(prob, y)
            loss_intensity = ce_loss(logits, i_cls)
            loss           = loss_cls + INTENSITY_LOSS_WEIGHT * loss_intensity

            if train and opt:
                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            total_loss += loss.item()
            preds = (prob > 0.5).float()
            correct += (preds == y).sum().item()
            total   += len(y)
            all_probs.extend(prob.detach().cpu().tolist())
            all_labels.extend(y.cpu().tolist())
            # Intensity accuracy
            i_pred = logits.argmax(dim=-1)
            intensity_correct += (i_pred == i_cls).sum().item()
            intensity_total   += len(i_cls)

    return {
        'loss':          total_loss / len(loader),
        'acc':           correct / total,
        'probs':         np.array(all_probs),
        'labels':        np.array(all_labels),
        'intensity_acc': intensity_correct / intensity_total,
    }


for epoch in range(1, FINETUNE_EPOCHS + 1):
    tr  = run_epoch_intensity(balanced_loader, train=True, opt=opt_full)
    vl  = run_epoch_intensity(val_loader, train=False)
    sched.step()

    preds = (vl['probs'] > 0.5).astype(int)
    tp = ((preds==1) & (vl['labels']==1)).sum()
    fp = ((preds==1) & (vl['labels']==0)).sum()
    fn = ((preds==0) & (vl['labels']==1)).sum()
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2   * prec * rec / (prec + rec + 1e-8)
    f2   = 5   * prec * rec / (4 * prec + rec + 1e-8)

    finetune_log.append({
        'epoch': epoch, 'train_loss': tr['loss'], 'val_loss': vl['loss'],
        'val_acc': vl['acc'], 'precision': prec, 'recall': rec,
        'f1': f1, 'f2': f2, 'intensity_acc': vl['intensity_acc'],
    })

    if f1 > best_f1:
        best_f1 = f1
        model.save(CKPT_DIR / 'finetuned_best.pt', {'epoch': epoch, 'f1': f1, 'precision': prec, 'recall': rec})
    if f2 > best_f2:
        best_f2 = f2
        model.save(CKPT_DIR / 'finetuned_f2_best.pt', {'epoch': epoch, 'f2': f2, 'precision': prec, 'recall': rec})

    if epoch % 5 == 0 or epoch == 1:
        print(f'  Epoch {epoch:3d}/{FINETUNE_EPOCHS} — loss:{vl["loss"]:.4f}  '
              f'F1:{f1:.3f}  F2:{f2:.3f}  P:{prec:.3f}  R:{rec:.3f}  '
              f'IntensityAcc:{vl["intensity_acc"]*100:.1f}%')

print(f'\nBest F1: {best_f1:.4f} → {CKPT_DIR / "finetuned_best.pt"}')
print(f'Best F2: {best_f2:.4f} → {CKPT_DIR / "finetuned_f2_best.pt"}')


---
## 6. Evaluation — NWS-Standard Verification Metrics

We use the same metrics NWS/SPC uses to verify severe weather forecasts:
- **POD** (Probability of Detection) — did we catch the tornadoes?
- **FAR** (False Alarm Ratio) — how often do we cry wolf?
- **CSI** (Critical Success Index) — combined skill score
- **AUC-ROC** — discrimination ability across all thresholds
- **Reliability / calibration** — does 30% probability mean 30% frequency?

In [ ]:
import pickle
from ml.model import TornadoTransformer
from ml.data import AtmosphericScaler, TornadoDataset
from sklearn.metrics import roc_auc_score

with open(DATA_DIR / 'cache/samples_2015_2025.pkl', 'rb') as f:
    _samples = pickle.load(f)
_scaler = AtmosphericScaler.load(CKPT_DIR / 'scaler.pkl')
_dataset = TornadoDataset(_samples, _scaler)
_, _val_ds = _dataset.split(val_frac=0.1)
_val_loader = DataLoader(_val_ds, batch_size=128, shuffle=False, num_workers=0)

def compute_metrics(probs, labels, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    tp = int(((preds==1) & (labels==1)).sum())
    fp = int(((preds==1) & (labels==0)).sum())
    fn = int(((preds==0) & (labels==1)).sum())
    tn = int(((preds==0) & (labels==0)).sum())
    pod  = tp / (tp + fn + 1e-8)
    far  = fp / (tp + fp + 1e-8)
    csi  = tp / (tp + fp + fn + 1e-8)
    bias = (tp + fp) / (tp + fn + 1e-8)
    f1   = 2*tp / (2*tp + fp + fn + 1e-8)
    f2   = 5*tp / (5*tp + 4*fn + fp + 1e-8)
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, POD=pod, FAR=far, CSI=csi,
                BIAS=bias, precision=tp/(tp+fp+1e-8), recall=pod, f1=f1, f2=f2)

def _eval_model(ckpt_name):
    m = TornadoTransformer.load(CKPT_DIR / ckpt_name, device=str(DEVICE)).to(DEVICE)
    m.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in _val_loader:
            x = batch['features'].to(DEVICE)
            y = batch['label'].to(DEVICE)
            out = m(x)
            prob = out['tornado_prob'].mean(dim=1)
            all_probs.extend(prob.cpu().tolist())
            all_labels.extend(y.cpu().tolist())
    return m, np.array(all_probs), np.array(all_labels).astype(int)

best_model, probs, labels = _eval_model('finetuned_best.pt')
f2_model,   probs_f2, _  = _eval_model('finetuned_f2_best.pt') if (CKPT_DIR / 'finetuned_f2_best.pt').exists() else (None, None, None)

m1 = compute_metrics(probs, labels)
auc1 = roc_auc_score(labels, probs)

print('A/B MODEL COMPARISON')
print(f'{"Metric":<12} {"F1-optimized":>14} {"F2-optimized":>14}')
print('-' * 42)
if probs_f2 is not None:
    m2 = compute_metrics(probs_f2, labels)
    auc2 = roc_auc_score(labels, probs_f2)
    for key, label in [('POD','POD'),('FAR','FAR'),('CSI','CSI'),('f1','F1'),('f2','F2')]:
        print(f'{label:<12} {m1[key]*100:>13.1f}% {m2[key]*100:>13.1f}%')
    print(f'{"AUC-ROC":<12} {auc1:>14.4f} {auc2:>14.4f}')
    print(f'{"TP":<12} {m1["tp"]:>14,} {m2["tp"]:>14,}')
    print(f'{"FP":<12} {m1["fp"]:>14,} {m2["fp"]:>14,}')
    print(f'{"FN":<12} {m1["fn"]:>14,} {m2["fn"]:>14,}')
else:
    print('F2 checkpoint not found — run Phase 2b to generate it.')
    for key, label in [('POD','POD'),('FAR','FAR'),('CSI','CSI'),('f1','F1')]:
        print(f'{label:<12} {m1[key]*100:>13.1f}%')
    print(f'{"AUC-ROC":<12} {auc1:>14.4f}')
    print(f'{"TP/FP/FN":<12} {m1["tp"]}/{m1["fp"]}/{m1["fn"]}')


In [ ]:
auc = auc1                           
best_thresh = 0.5


fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Model Evaluation — TornadoTransformer', fontsize=14)

# 1. ROC Curve
ax = axes[0, 0]
fpr, tpr, roc_thresh = roc_curve(labels, probs)
ax.plot(fpr, tpr, color='#38bdf8', linewidth=2, label=f'AUC = {auc:.3f}')
ax.plot([0,1],[0,1], '--', color='gray', alpha=0.5, label='Random')
ax.fill_between(fpr, tpr, alpha=0.1, color='#38bdf8')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend()
ax.grid(alpha=0.2)

# 2. Precision-Recall Curve
ax = axes[0, 1]
prec_arr, rec_arr, pr_thresh = precision_recall_curve(labels, probs)
ax.plot(rec_arr, prec_arr, color='#f59e0b', linewidth=2)
ax.fill_between(rec_arr, prec_arr, alpha=0.1, color='#f59e0b')
ax.set_xlabel('Recall (POD)')
ax.set_ylabel('Precision')
ax.set_title('Precision–Recall Curve')
ax.grid(alpha=0.2)

# 3. Reliability (Calibration) Curve
ax = axes[0, 2]
prob_true, prob_pred = calibration_curve(labels, probs, n_bins=10, strategy='quantile')
ax.plot(prob_pred, prob_true, 'o-', color='#10b981', linewidth=2, label='Model')
ax.plot([0,1],[0,1], '--', color='gray', alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Reliability Diagram')
ax.legend()
ax.grid(alpha=0.2)

# 4. CSI by threshold
ax = axes[1, 0]
thresholds = np.linspace(0.05, 0.95, 50)
csi_vals = [compute_metrics(probs, labels, t)['CSI'] for t in thresholds]
pod_vals = [compute_metrics(probs, labels, t)['POD'] for t in thresholds]
far_vals = [compute_metrics(probs, labels, t)['FAR'] for t in thresholds]
ax.plot(thresholds, csi_vals, color='#a78bfa', linewidth=2, label='CSI')
ax.plot(thresholds, pod_vals, color='#10b981', linewidth=2, label='POD')
ax.plot(thresholds, far_vals, color='#ef4444', linewidth=2, label='FAR')
best_thresh = thresholds[np.argmax(csi_vals)]
ax.axvline(best_thresh, color='white', linestyle='--', alpha=0.5, label=f'Best thresh={best_thresh:.2f}')
ax.set_xlabel('Decision Threshold')
ax.set_title('CSI / POD / FAR by Threshold')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

# 5. Confusion matrix
ax = axes[1, 1]
cm = confusion_matrix(labels, (probs >= best_thresh).astype(int))
sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
            xticklabels=['No Tornado','Tornado'], yticklabels=['No Tornado','Tornado'])
ax.set_title(f'Confusion Matrix (thresh={best_thresh:.2f})')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')

# 6. Training history (only available if run in same session as training)
ax = axes[1, 2]
if 'finetune_log' in dir() and finetune_log:
    log_df = pd.DataFrame(finetune_log)
    ax.plot(log_df['epoch'], log_df['f1'],        color='#a78bfa', linewidth=2, label='F1')
    ax.plot(log_df['epoch'], log_df['precision'], color='#38bdf8', linewidth=2, label='Precision')
    ax.plot(log_df['epoch'], log_df['recall'],    color='#10b981', linewidth=2, label='Recall')
    ax.axvline(log_df.loc[log_df['f1'].idxmax(), 'epoch'], color='white', linestyle='--', alpha=0.4)
    ax.legend()
else:
    ax.text(0.5, 0.5, 'Training history\nnot available', ha='center', va='center',
            transform=ax.transAxes, color='gray')
ax.set_xlabel('Epoch')
ax.set_title('Fine-tune Metrics Over Time')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(DATA_DIR / 'eval_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nOptimal threshold for CSI: {best_thresh:.2f}')
best_metrics = compute_metrics(probs, labels, best_thresh)
print(f'  At optimal threshold → POD:{best_metrics["POD"]*100:.1f}%  FAR:{best_metrics["FAR"]*100:.1f}%  CSI:{best_metrics["CSI"]*100:.1f}%')

---
## 7. Benchmark — vs. Statistical Fallback & SPC Day-1 Outlook

Compare the trained model against:
1. Our own STP-based statistical predictor (current fallback)
2. Published NWS tornado warning verification stats (historical baseline)

In [ ]:
from ml.inference import TornadoPredictor

stat_predictor = TornadoPredictor(model_path=Path('/nonexistent'), scaler_path=CKPT_DIR/'scaler.pkl')
stat_probs = []
for s in _val_ds.samples:
    feats = _scaler.transform(s.features)
    p1, p2, _ = stat_predictor._statistical_predict(feats)
    stat_probs.append(max(p1, p2))
stat_probs  = np.array(stat_probs)
stat_labels = np.array([s.label for s in _val_ds.samples])

nws_baseline = {'POD': 0.72, 'FAR': 0.73, 'CSI': 0.23, 'lead_time_min': 13}
model_m = compute_metrics(probs, labels)
stat_m  = compute_metrics(stat_probs, stat_labels)

print('BENCHMARK COMPARISON')
print(f'{"Metric":<12} {"NWS Baseline":>14} {"Statistical":>14} {"TornadoTransformer":>20}')
print('-' * 65)
for metric in ['POD', 'FAR', 'CSI']:
    print(f'{metric:<12} {nws_baseline[metric]*100:>13.1f}% {stat_m[metric]*100:>13.1f}% {model_m[metric]*100:>19.1f}%')
print(f'{"AUC-ROC":<12} {"—":>14} {roc_auc_score(stat_labels, stat_probs):>13.3f} {auc1:>20.3f}')
print(f'{"Lead time":<12} {"13 min":>14} {"120 min":>14} {"120 min":>20}')
print('\nNote: NWS baseline is for specific tornado warnings (avg 13-min lead).')
print('Our model targets 2-hour coordinate prediction — a fundamentally harder task.')

---
## 8. Export Checkpoint for Serving

The app at `backend/app/services/prediction_service.py` loads from `checkpoints/finetuned_best.pt` automatically. Just confirm it's saved.

In [ ]:
ckpt_path = CKPT_DIR / 'finetuned_best.pt'
assert ckpt_path.exists(), f'Checkpoint not found at {ckpt_path}'

# Verify round-trip
reloaded = TornadoTransformer.load(ckpt_path, device='cpu')
reloaded.eval()
test_input = torch.randn(1, 24, N_FEATURES)
with torch.no_grad():
    out = reloaded(test_input)

print(f'Checkpoint: {ckpt_path}  ({ckpt_path.stat().st_size/1e6:.1f} MB)')
print(f'Output shape — prob: {out["tornado_prob"].shape}  intensity: {out["intensity"].shape}')
print(f'Sample prediction — 1h: {out["tornado_prob"][0,0].item():.4f}  2h: {out["tornado_prob"][0,1].item():.4f}')
print()
print('Model is ready. Restart the backend server to pick up the new checkpoint:')
print('  cd backend && uvicorn app.main:app --reload --port 8000')

In [ ]:
# ── Held-out year validation: 2025 ──────────────────────────────────────────
# 2025 was never seen during training or checkpoint selection.
import pickle
from ml.data import AtmosphericScaler, TornadoDataset
from ml.model import TornadoTransformer, INTENSITY_CLASS_NAMES
from sklearn.metrics import roc_auc_score

_scaler_test = AtmosphericScaler.load(CKPT_DIR / 'scaler.pkl')
_ds_test   = TornadoDataset(test_samples, _scaler_test)
_loader_test = DataLoader(_ds_test, batch_size=128, shuffle=False, num_workers=0)

def _eval_ckpt(ckpt_name, loader):
    m = TornadoTransformer.load(CKPT_DIR / ckpt_name, device=str(DEVICE)).to(DEVICE)
    m.eval()
    all_probs, all_labels, all_logits, all_icls = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            x  = batch['features'].to(DEVICE)
            y  = batch['label'].to(DEVICE)
            ef = batch['ef_scale'].to(DEVICE)
            out = m(x)
            prob   = out['tornado_prob'].mean(dim=1)
            logits = out['intensity_logits'].mean(dim=1)
            all_probs.extend(prob.cpu().tolist())
            all_labels.extend(y.cpu().tolist())
            all_logits.append(logits.cpu())
            all_icls.extend(ef_to_intensity_class(ef, y).cpu().tolist())
    logits_all = torch.cat(all_logits)
    return (np.array(all_probs), np.array(all_labels).astype(int),
            logits_all, np.array(all_icls))

p_test, l_test, logits_test, icls_test = _eval_ckpt('finetuned_best.pt', _loader_test)
auc_test = roc_auc_score(l_test, p_test)
m_test   = compute_metrics(p_test, l_test)

# Intensity classification accuracy on test set
i_pred_test = logits_test.argmax(dim=-1).numpy()
intensity_acc = (i_pred_test == icls_test).mean()
per_class_acc = [
    (i_pred_test[icls_test==c] == c).mean() if (icls_test==c).any() else float('nan')
    for c in range(3)
]

print('=' * 55)
print(f'HELD-OUT {TEST_YEAR} VALIDATION (never seen during training)')
print('=' * 55)
print(f'  Binary Detection:')
print(f'    AUC-ROC: {auc_test:.4f}')
print(f'    POD:     {m_test["POD"]*100:.1f}%')
print(f'    FAR:     {m_test["FAR"]*100:.1f}%')
print(f'    CSI:     {m_test["CSI"]*100:.1f}%')
print(f'    F1:      {m_test["f1"]*100:.1f}%')
print()
print(f'  Intensity Classification (3-class):')
print(f'    Overall accuracy: {intensity_acc*100:.1f}%')
for c, name in enumerate(INTENSITY_CLASS_NAMES):
    n = (icls_test==c).sum()
    print(f'    {name}: {per_class_acc[c]*100:.1f}%  (n={n:,})')


In [ ]:
# ── Threshold Calibration (self-contained) ───────────────────────────────────
# Find optimal decision threshold on val set, apply to 2019 held-out test.
import sys, pickle, json as _json
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

sys.path.insert(0, "../backend")
DATA_DIR = Path("../backend/data")
CKPT_DIR = Path("../backend/checkpoints")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

from ml.model import TornadoTransformer
from ml.data import AtmosphericScaler, TornadoDataset

def compute_metrics(probs, labels, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    tp = int(((preds==1) & (labels==1)).sum())
    fp = int(((preds==1) & (labels==0)).sum())
    fn = int(((preds==0) & (labels==1)).sum())
    tn = int(((preds==0) & (labels==0)).sum())
    pod = tp / (tp + fn + 1e-8)
    far = fp / (tp + fp + 1e-8)
    csi = tp / (tp + fp + fn + 1e-8)
    f1  = 2*tp / (2*tp + fp + fn + 1e-8)
    f2  = 5*tp / (5*tp + 4*fn + fp + 1e-8)
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, POD=pod, FAR=far, CSI=csi,
                precision=tp/(tp+fp+1e-8), recall=pod, f1=f1, f2=f2)

def _run_model(ckpt_name, loader):
    m = TornadoTransformer.load(CKPT_DIR / ckpt_name, device=str(DEVICE)).to(DEVICE)
    m.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            x = batch["features"].to(DEVICE)
            y = batch["label"]
            out = m(x)
            prob = out["tornado_prob"].mean(dim=1)
            all_probs.extend(prob.cpu().tolist())
            all_labels.extend(y.tolist())
    return np.array(all_probs), np.array(all_labels).astype(int)

# Load val set (2015-2025, last 10%)
_scaler = AtmosphericScaler.load(CKPT_DIR / "scaler.pkl")
with open(DATA_DIR / "cache/samples_2015_2025.pkl", "rb") as f:
    _samples_tv = pickle.load(f)
_val_ds = TornadoDataset(_samples_tv, _scaler).split(val_frac=0.1)[1]
_val_loader = DataLoader(_val_ds, batch_size=128, shuffle=False, num_workers=0)

# Load 2019 test set
with open(DATA_DIR / "cache/samples_2019_2019.pkl", "rb") as f:
    _samples_2019 = pickle.load(f)
_ds_2019 = TornadoDataset(_samples_2019, _scaler)
_loader_2019 = DataLoader(_ds_2019, batch_size=128, shuffle=False, num_workers=0)

print("Evaluating val set...")
probs_val, labels_val = _run_model("finetuned_best.pt", _val_loader)
print("Evaluating 2019 held-out set...")
p_2019, l_2019 = _run_model("finetuned_best.pt", _loader_2019)
auc_2019 = roc_auc_score(l_2019, p_2019)

# Sweep 200 thresholds on val set
thresholds = np.linspace(0.01, 0.99, 200)
best_thresh_csi, best_csi_val = 0.5, 0.0
best_thresh_f2,  best_f2_val  = 0.5, 0.0
for t in thresholds:
    mv = compute_metrics(probs_val, labels_val, t)
    if mv["CSI"] > best_csi_val:
        best_csi_val, best_thresh_csi = mv["CSI"], t
    if mv["f2"] > best_f2_val:
        best_f2_val, best_thresh_f2 = mv["f2"], t

print(f"\nOptimal threshold (CSI): {best_thresh_csi:.3f}  ->  val CSI: {best_csi_val*100:.1f}%")
print(f"Optimal threshold (F2):  {best_thresh_f2:.3f}  ->  val F2:  {best_f2_val*100:.1f}%")

# Apply calibrated thresholds to 2019 held-out
m_05  = compute_metrics(p_2019, l_2019, 0.5)
m_csi = compute_metrics(p_2019, l_2019, best_thresh_csi)
m_f2  = compute_metrics(p_2019, l_2019, best_thresh_f2)

print()
print("2019 HELD-OUT: THRESHOLD COMPARISON")
print(f"{'Metric':<10} {'thresh=0.5':>12} {'thresh=CSI':>12} {'thresh=F2':>12}")
print("-" * 50)
for key, lbl in [("POD","POD"),("FAR","FAR"),("CSI","CSI"),("f1","F1"),("f2","F2")]:
    print(f"{lbl:<10} {m_05[key]*100:>11.1f}% {m_csi[key]*100:>11.1f}% {m_f2[key]*100:>11.1f}%")
print(f"{'AUC-ROC':<10} {auc_2019:>12.4f} {auc_2019:>12.4f} {auc_2019:>12.4f}")

print()
print("vs NWS operational (13-min lead):")
print(f"  POD: NWS 72%  ->  best 2019: {max(m_csi['POD'], m_f2['POD'])*100:.1f}%")
print(f"  FAR: NWS 73%  ->  best 2019: {min(m_csi['FAR'], m_f2['FAR'])*100:.1f}%")
print(f"  CSI: NWS 23%  ->  best 2019: {max(m_csi['CSI'], m_f2['CSI'])*100:.1f}%")

thresh_cfg = {"csi_threshold": float(best_thresh_csi), "f2_threshold": float(best_thresh_f2)}
thresh_path = CKPT_DIR / "thresholds.json"
with open(thresh_path, "w") as f:
    _json.dump(thresh_cfg, f, indent=2)
print(f"\nThresholds saved to {thresh_path}")


---
## Next Steps

| Goal | Action |
|------|--------|
| Improve CSI | Add NOAA sounding data for true SRH/LCL (currently estimated) |
| Better calibration | Post-hoc Platt scaling or isotonic regression on val set |
| Spatial context | Add neighboring grid-point features (storm motion, mesoscale convective info) |
| Longer lead time | Extend `forecast_hours` in `ModelConfig` to 4h, retrain |
| Larger model | Increase `d_model=512`, `n_encoder_layers=8` — needs GPU |
| Real-time SRH | Integrate NOAA IEM sounding API for actual wind profiles |